In [ ]:
import pandas as pd
import numpy as np
import re
import html
from bs4 import BeautifulSoup
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from catboost import CatBoostRegressor

# ==============================
# 1. Load Dataset
# ==============================
df = pd.read_csv("/Users/bhavinbaldota/Library/CloudStorage/OneDrive-MSFT/Desktop/Amazon L1/dataset/train.csv")
pd.set_option('display.max_colwidth', 200)

print("\n=== Sample Rows ===")
print(df.head(5))

# ==============================
# 2. Inspect Structure
# ==============================
patterns = ["item name:", "value:", "unit:"]
for pat in patterns:
    pct = df['catalog_content'].str.contains(pat, case=False, na=False).mean()
    print(f"{pat} found in {pct*100:.1f}% of rows")

# ==============================
# 3. Text Cleaning
# ==============================
def clean_text(text):
    """Remove HTML, emojis, special chars, lowercase."""
    if pd.isna(text):
        return ""
    text = html.unescape(text)
    text = BeautifulSoup(text, "lxml").get_text(separator=" ")
    text = text.encode("ascii", "ignore").decode()
    text = text.lower()
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['catalog_clean'] = df['catalog_content'].apply(clean_text)

# ==============================
# 4. General Quantity Extraction
# ==============================
def extract_quantity(text):
    match = re.search(r'(\d+(?:\.\d+)?)\s?(oz|ml|g|kg|lb|count)', text)
    if match:
        value = float(match.group(1))
        unit = match.group(2)
        return value, unit
    return np.nan, ""

df[['value', 'unit']] = df['catalog_clean'].apply(lambda t: pd.Series(extract_quantity(t)))

# ==============================
# 5. Pack Size Extraction
# ==============================
df['pack_size'] = df['catalog_clean'].str.extract(r'pack\s?of\s?(\d+)', re.IGNORECASE).fillna(1).astype(int)

# ==============================
# 6. Total Quantity Feature
# ==============================
unit_map = {
    "ounce": 28.3495, "oz": 28.3495,
    "fl oz": 29.5735, "fluid ounce": 29.5735,
    "count": 1, "ml": 1, "g": 1, "kg": 1000, "lb": 453.592
}
df['normalized_value'] = df.apply(
    lambda row: row['value'] * unit_map.get(row['unit'], 1) if not pd.isna(row['value']) else np.nan,
    axis=1
)
df['total_quantity'] = df['normalized_value'] * df['pack_size']
df['log_total_quantity'] = np.log1p(df['total_quantity'].fillna(0))

# ==============================
# 7. Brand Extraction
# ==============================
df['brand'] = df['catalog_clean'].str.split().str[0].fillna("unknown")
df['brand_popularity'] = df.groupby('brand')['brand'].transform('count')

# ==============================
# 8. Sentence-BERT Embeddings
# ==============================
print("\nGenerating Sentence-BERT embeddings...")
model_bert = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model_bert.encode(df['catalog_clean'], batch_size=64, show_progress_bar=True)

print("Reducing embedding dimensions...")
pca = PCA(n_components=100)
embeddings_pca = pca.fit_transform(embeddings)
embeddings_df = pd.DataFrame(embeddings_pca, index=df.index)

# ==============================
# 9. Final Feature Set
# ==============================
feature_df = pd.concat([
    df[['log_total_quantity', 'pack_size', 'brand_popularity']],
    embeddings_df
], axis=1).fillna(0)

# ==============================
# 10. Train/Test Split
# ==============================
X_train, X_val, y_train, y_val = train_test_split(
    feature_df, np.log1p(df['price']), test_size=0.2, random_state=42
)

# ==============================
# 11. SMAPE Metric
# ==============================
def smape(y_true, y_pred):
    y_true = np.expm1(y_true)  # back-transform
    y_pred = np.expm1(y_pred)
    return np.mean(
        np.abs(y_pred - y_true) / ((np.abs(y_true) + np.abs(y_pred)) / 2)
    ) * 100

# ==============================
# 12. Train CatBoost
# ==============================
print("\nTraining CatBoost...")
model = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.05,
    depth=8,
    loss_function='RMSE',
    random_seed=42,
    verbose=200
)
model.fit(X_train, y_train)

# ==============================
# 13. Evaluate
# ==============================
val_preds = model.predict(X_val)
print("Validation SMAPE:", smape(y_val, val_preds))


=== Sample Rows ===
   sample_id  \
0      33127   
1     198967   
2     261251   
3      55858   
4     292686   

                                                                                                                                                                                           catalog_content  \
0                                                                                                           Item Name: La Victoria Green Taco Sauce Mild, 12 Ounce (Pack of 6)\nValue: 72.0\nUnit: Fl Oz\n   
1  Item Name: Salerno Cookies, The Original Butter Cookies, 8 Ounce (Pack of 4)\nBullet Point 1: Original Butter Cookies: Classic butter cookies made with real butter\nBullet Point 2: Variety Pack: I...   
2  Item Name: Bear Creek Hearty Soup Bowl, Creamy Chicken with Rice, 1.9 Ounce (Pack of 6)\nBullet Point 1: Loaded with hearty long grain wild rice and vegetables\nBullet Point 2: Full of hearty good...   
3  Item Name: Judee’s Blue Cheese Powder 11.25 oz - Gluten

Batches:   0%|          | 0/1172 [00:00<?, ?it/s]